# ITNPA4-34 Project 1 — Questions 3 & 4: Topic Analysis, Summarisation, LDA & pyLDAvis
**Module:** Natural Language Processing with Python (`ITNPA4-34`, NQF Level 8)  
**Student:** Resego Ngwenya (`EDUV4824738`) | **Campus:** Online | **Lecturer:** Mr. T. Mavetera  

This notebook ingests `research_corpus.zip` into a structured DataFrame (Q3.1), applies dual-stage denoising and bigram phrase detection (Q3.2), summarises the first article using TextRank graph centrality (Q3.3), synthesises Articles 2–5 using aggregate LSA + Maximal Marginal Relevance (Q3.4), tunes and interprets Latent Dirichlet Allocation (Q4.1), and builds an interactive `pyLDAvis` Intertopic Distance Map (Q4.2).

In [1]:
import json
import os
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import pyLDAvis
import pyLDAvis.gensim_models
import seaborn as sns
from gensim import corpora
from gensim.models import CoherenceModel, LdaModel, Phrases
from gensim.models.phrases import Phraser
from wordcloud import WordCloud

ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT_DIR / "scripts")) if "sys" in globals() else None
import sys
sys.path.insert(0, str(ROOT_DIR / "scripts"))
from run_q3_q4_pipeline import (
    ARTICLE_META, DOMAIN_ACADEMIC_STOPWORDS, build_passages,
    extract_clean_sentences, lsa_mmr_summarise, textrank_summarise,
    tokenize_for_topics, unwrap_and_clean_prose
)
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
sys.path.insert(0, str(ROOT_DIR / "deployment"))
from text_preprocessor import ensure_nltk_resources

ensure_nltk_resources()
print("Loaded corpus utilities cleanly from", ROOT_DIR)

Loaded corpus utilities cleanly from C:\Users\resego.ngwenya\.gemini\antigravity\scratch\ITNPA4-34_Project


### Environment & Shared Utilities Loaded
Shared prose-unwrapping, citation-stripping, TextRank, LSA+MMR, and passage-segmentation functions are imported from the verified project pipeline.

In [2]:
# Question 3.1 & 3.2: Load research_corpus.zip into DataFrame and apply dual-stage denoising
records = []
with zipfile.ZipFile(ROOT_DIR / "datasets" / "research_corpus.zip", "r") as zf:
    for doc_num, member in enumerate(sorted(m for m in zf.namelist() if m.endswith(".txt")), start=1):
        fname = os.path.basename(member)
        raw_txt = zf.read(member).decode("utf-8", errors="replace")
        records.append({
            "doc_id": doc_num,
            "file_name": fname,
            "title": ARTICLE_META[fname]["title"],
            "raw_words": len(raw_txt.split()),
            "raw_text": raw_txt,
        })

df_corpus = pd.DataFrame(records)
df_corpus["clean_prose"] = df_corpus["raw_text"].apply(unwrap_and_clean_prose)
df_corpus["sentences"] = df_corpus["clean_prose"].apply(extract_clean_sentences)
df_corpus["prose_dedup"] = df_corpus["sentences"].apply(" ".join)

stop_words = set(stopwords.words("english")) | DOMAIN_ACADEMIC_STOPWORDS
lemmatizer = WordNetLemmatizer()
unigrams = [tokenize_for_topics(t, stop_words, lemmatizer) for t in df_corpus["prose_dedup"]]
phraser = Phraser(Phrases(unigrams, min_count=4, threshold=8.0, delimiter="_"))
phrased_docs = [phraser[d] for d in unigrams]
df_corpus["clean_tokens"] = phrased_docs
df_corpus["clean_token_count"] = [len(d) for d in phrased_docs]
df_corpus["sentence_count"] = df_corpus["sentences"].apply(len)

df_corpus[["doc_id", "file_name", "title", "raw_words", "sentence_count", "clean_token_count"]]

,doc_id,file_name,title,raw_words,sentence_count,clean_token_count
0,1,research_1.txt,Mixed Methods Research: A Research Paradigm Wh...,5602,80,1315
1,2,research_2.txt,"Now I See It, Now I Don't: Researcher's Positi...",2416,59,1074
2,3,research_3.txt,Developmental Research: Studies of Instruction...,3828,154,1713
3,4,research_4.txt,The Use of Cronbach's Alpha When Developing an...,5922,183,2650
4,5,research_5.txt,Denzin's Paradigm Shift: Revisiting Triangulat...,4027,154,1702


### Analysis of Q3.1 & Q3.2 Corpus Structuring and Denoising
All five methodology articles (`research_1.txt`–`research_5.txt`) were parsed successfully. Dual-stage denoising repaired broken PDF line wraps, removed parenthetical APA citations and numerical clutter while preserving prose punctuation for sentence segmentation, and merged key multi-word technical expressions via Gensim bigram collocations (`mixed_methods`, `cronbach_alpha`, `qualitative_research`, `data_saturation`).

In [3]:
# Question 3.3: TextRank Extractive Summarisation of Article 1 (research_1.txt)
art1_sents = df_corpus.loc[0, "sentences"]
top_idx_1, tr_scores, tr_graph = textrank_summarise(art1_sents, top_n=6)
for rank_i, idx_s in enumerate(top_idx_1, 1):
    print(f"[{rank_i}] Sentence #{idx_s+1} (PageRank={tr_scores[idx_s]:.4f}):\n    {art1_sents[idx_s]}\n")

[1] Sentence #1 (PageRank=0.0441):
    The purposes of this article are to position mixed methods research (mixed research is a synonym) as the natural complement to traditional qualitative and quantitative research, to present pragmatism as offering an attractive philosophical partner for mixed methods research, and to provide a framework for designing and conducting mixed methods research.

[2] Sentence #2 (PageRank=0.0305):
    In doing this, we briefly review the paradigm "wars" and incompatibility thesis, we show some commonalities between quantitative and qualitative research, we explain the tenets of pragmatism, we explain the fundamental principle of mixed research and how to apply it, we provide specific sets of designs for the two major types of mixed methods research (mixed-model designs and mixed-method designs), and, finally, we explain mixed methods research as following (recursively) an eight-step process.

[3] Sentence #22 (PageRank=0.0358):
    Our purpose in writing t

### Analysis of Q3.3 TextRank Summary
TextRank builds a weighted cosine-similarity graph over the cleaned sentences of `research_1.txt` and extracts the 6 sentences with the highest PageRank eigenvector centrality, faithfully capturing mixed-methods philosophical pragmatism, compatibility, and pluralism in chronological reading order.

In [4]:
# Question 3.4: Aggregate Multi-Document LSA + MMR Summarisation (Articles 2-5)
agg_sents, agg_docs = [], []
for i in range(1, len(df_corpus)):
    for s in df_corpus.loc[i, "sentences"]:
        agg_sents.append(s)
        agg_docs.append(df_corpus.loc[i, "file_name"])

sel_idx, lsa_sal, U_lsa, svd_agg = lsa_mmr_summarise(agg_sents, agg_docs, n_components=4, top_n=8, lam=0.68, max_per_doc=2)
agg_df = pd.DataFrame([
    {"Rank": r + 1, "Source": agg_docs[idx], "LSA_Saliency": round(float(lsa_sal[idx]), 4), "Sentence": agg_sents[idx]}
    for r, idx in enumerate(sel_idx)
])
agg_df

,Rank,Source,LSA_Saliency,Sentence
0,1,research_3.txt,1.0000,"In its simplest form, developmental research c..."
1,2,research_5.txt,0.9500,Denzin (1989) suggested data triangulation for...
2,3,research_4.txt,0.7982,The Frequency of Use of Cronbach's Alpha in Sc...
3,4,research_3.txt,0.7672,The purposes of this chapter are to explore th...
4,5,research_4.txt,0.7550,Adadan and Savasci call upon the Cronbach alph...
5,6,research_5.txt,0.8617,"The Triangulation Confusion Frequently, we see..."
6,7,research_2.txt,0.5480,One goal of reflexivity in qualitative researc...
7,8,research_2.txt,0.4418,Because reflexivity is a major strategy for qu...


### Analysis of Q3.4 Multi-Document Aggregate Summary
Unlike single-document TextRank, **LSA (`TruncatedSVD`) + Maximal Marginal Relevance (MMR)** decomposes the combined sentence-term matrix of Articles 2–5 into orthogonal latent semantic concepts while penalising near-duplicate sentences. Consequently, the aggregate summary spans reflexivity (`research_2.txt`), developmental instructional research (`research_3.txt`), Cronbach's alpha reliability (`research_4.txt`), and triangulation/data saturation (`research_5.txt`).

In [5]:
# Question 4.1 & 4.2: Passage-Segmented LDA Coherence Sweep + Topic/Document Distributions
all_sents = [s for slist in df_corpus["sentences"] for s in slist]
all_docs = [fn for fn, slist in zip(df_corpus["file_name"], df_corpus["sentences"]) for _ in slist]
passages = build_passages(all_sents, all_docs, target_words=150)
passage_tokens = [phraser[tokenize_for_topics(t, stop_words, lemmatizer)] for _, t in passages]

id2word = corpora.Dictionary(passage_tokens)
id2word.filter_extremes(no_below=2, no_above=0.65)
bow_corpus = [id2word.doc2bow(toks) for toks in passage_tokens]

q3_q4_meta = json.loads((ROOT_DIR / "outputs" / "q3_q4_results.json").read_text(encoding="utf-8"))
opt_k = q3_q4_meta["q4_1_lda"]["optimal_k"]
print(f"Passages: {len(passages)} | Filtered Dictionary: {len(id2word)} terms | Optimal Topics (C_v peak): {opt_k}")
display_sweep = pd.DataFrame(q3_q4_meta["q4_1_lda"]["coherence_sweep"]).round(4)
print(display_sweep.to_string(index=False))

doc_topic_csv = pd.read_csv(ROOT_DIR / "outputs" / "q4_doc_topic_distribution.csv")
doc_topic_csv

Passages: 111 | Filtered Dictionary: 1161 terms | Optimal Topics (C_v peak): 3


 k  coherence_cv  log_perplexity  perplexity
 3        0.5456         -6.7009    813.1598
 4        0.4608         -6.7652    867.0995
 5        0.4788         -6.7671    868.7770
 6        0.4103         -6.9601   1053.7428
 7        0.3793         -7.0249   1124.2475
 8        0.3807         -7.0565   1160.3755


,Article,Topic 1,Topic 2,Topic 3,Dominant_Topic
0,Article 1: Mixed Methods,0.008462,0.867158,0.124380,Topic 2
1,Article 2: Reflexivity,0.000571,0.952903,0.046526,Topic 2
2,Article 3: Developmental Research,0.000254,0.049778,0.949968,Topic 3
3,Article 4: Cronbach's Alpha,0.004853,0.000195,0.994952,Topic 3
4,Article 5: Triangulation,0.570051,0.429666,0.000283,Topic 1


### Conclusion & Interactive pyLDAvis Visualisation (Q4.1–Q4.2)
Segmenting the 5 methodology articles into coherent passages produces meaningful word co-occurrence statistics and strong $C_v$ coherence. Each discovered LDA topic maps directly onto a substantive methodology theme across the five documents. The interactive HTML visualisation generated via `pyLDAvis.gensim_models.prepare(..., mds='mmds')` is saved at `outputs/pyldavis_intertopic_map.html`, alongside publication figures in `figures/`.